In [101]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error, 
    mean_squared_error,
    root_mean_squared_error, 
    mean_absolute_percentage_error,
    make_scorer
)

from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

from joblib import dump, load
import os

## Dataset import

In [ ]:
df = pd.read_csv("../data/processed/bc_clean.csv")
df = df.drop(["Unnamed: 0"], axis=1)

In [103]:
X = df.drop(["price"], axis=1)
Y = df["price"]

### Linear models

#### Train test split

In [104]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42, shuffle=True
)

#### Pipeline

The class below will be applied to columns containing values = 0 after the imputation. It'll be used inside a column transformer

In [15]:
class StandardizeIgnoreZero(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, Y=None):
        X = np.asarray(X, dtype=float).copy()
        self.means_ = []
        self.stds_ = []

        for i in range(X.shape[1]):
            mask = X[:, i] != 0
            mean = X[mask, i].mean()
            std = X[mask, i].std()
    
            self.means_.append(mean)
            self.stds_.append(std)
        
        return self

    def transform(self, X, Y=None):
        X = np.asarray(X, dtype=float).copy()
        for i in range(X.shape[1]):
            mask = X[:, i] != 0
            X[mask, i] = (X[mask, i] - self.means_[i]) / self.stds_[i]
        return X

In [16]:
numeric_col = ["latitude", "longitude", "property-beds", "property-baths"]
num_col_ignore_zero = ["Acreage", "Property Tax", "Square Footage"]

ct = ColumnTransformer(
    transformers = [
        ("std_ignore_zeros", StandardizeIgnoreZero(), num_col_ignore_zero),
        ("std", StandardScaler(), numeric_col)
    ],
    remainder = "passthrough"
)

In [17]:
pipe = Pipeline([
    ("col_transform", ct),
    ("model", LinearRegression())
])

In [ ]:
# I will not use a SVR model due to the computation time that increases a lot as the number of columns increases.

param_grid = [
    {
        "model" : [LinearRegression()]
    }, 
    {
        "model" : [Ridge()],
        "model__alpha" : [0.001, 0.01, 0.1, 1, 10],
        "model__max_iter" : [3000]
    },
    {
        "model" : [Lasso()],
        "model__alpha" : [0.001, 0.01, 0.1, 1, 10],
        "model__max_iter" : [3000]

    },
    {
        "model" : [ElasticNet()],
        "model__alpha" : [0.001, 0.01, 0.1, 1, 10],
        "model__l1_ratio" : [0.1, 0.5, 0.9],
        "model__max_iter" : [3000]

    },  
    {
        "model" : [KNeighborsRegressor()],
        "model__n_neighbors" : np.arange(5, 51, 2),
        "model__p" : [1, 2],
        "model__weights" : ["uniform", "distance"]
    }
]

In [62]:
scoring = {
    "r2_score" : make_scorer(r2_score, greater_is_better=True),
    "neg_mae" : make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_mse" : make_scorer(mean_squared_error, greater_is_better=False),
    "neg_rmse" : make_scorer(root_mean_squared_error, greater_is_better=False),
    "neg_mape" : make_scorer(mean_absolute_percentage_error, greater_is_better=False),
}

In [63]:
grid = GridSearchCV(
    estimator=pipe,
    param_grid = param_grid,
    scoring = scoring["neg_rmse"],
    cv = KFold(5),
    n_jobs = 10,
    verbose = 4
)

In [64]:
if os.path.isfile("../artifacts/grid_linear.pkl"):
    print("la grille existe déjà")
else : 
    grid.fit(X_train, Y_train)
    dump(grid, "../artifacts/grid_linear.pkl")

Fitting 5 folds for each of 118 candidates, totalling 590 fits


/home/marwane/canada-housing-predictor/venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.142e+14, tolerance: 4.958e+12
  model = cd_fast.enet_coordinate_descent(
/home/marwane/canada-housing-predictor/venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.285e+14, tolerance: 5.058e+12
  model = cd_fast.enet_coordinate_descent(
/home/marwane/canada-housing-predictor/venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the sca

[CV 2/5] END ...model=LinearRegression();, score=-1312449.540 total time=   0.1s
[CV 4/5] END model=Ridge(), model__alpha=0.01, model__max_iter=3000;, score=-1693628.556 total time=   0.1s
[CV 1/5] END model=Ridge(), model__alpha=1, model__max_iter=3000;, score=-1093863.646 total time=   0.1s
[CV 1/5] END model=Lasso(), model__alpha=0.001, model__max_iter=3000;, score=-1094337.703 total time=  11.3s
[CV 5/5] END model=Lasso(), model__alpha=1, model__max_iter=3000;, score=-1446024.989 total time=  10.3s
[CV 1/5] END model=ElasticNet(), model__alpha=0.001, model__l1_ratio=0.5, model__max_iter=3000;, score=-1092816.404 total time=   5.7s
[CV 5/5] END model=ElasticNet(), model__alpha=0.01, model__l1_ratio=0.1, model__max_iter=3000;, score=-1448780.009 total time=   0.3s
[CV 3/5] END model=ElasticNet(), model__alpha=0.01, model__l1_ratio=0.5, model__max_iter=3000;, score=-1204035.639 total time=   0.6s
[CV 4/5] END model=ElasticNet(), model__alpha=0.01, model__l1_ratio=0.9, model__max_iter=

In [65]:
grid = load("../artifacts/grid_linear.pkl")

In [66]:
grid.best_params_

{'model': KNeighborsRegressor(),
 'model__n_neighbors': np.int64(7),
 'model__p': 1,
 'model__weights': 'distance'}

In [67]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("R²:", r2_score(Y_test, y_pred))
print("MAE:", mean_absolute_error(Y_test, y_pred))
print("RMSE:", root_mean_squared_error(Y_test, y_pred))
print("MAPE:", mean_absolute_percentage_error(Y_test, y_pred))

R²: 0.6894406351107201
MAE: 405551.0159064908
RMSE: 964569.2747073396
MAPE: 0.23733243470514692


In [79]:
results = pd.DataFrame(grid.cv_results_)[["param_model", "rank_test_score", "mean_test_score", "std_test_score"]]
results.sort_values("rank_test_score").iloc[:20,:]

,param_model,rank_test_score,mean_test_score,std_test_score
31,KNeighborsRegressor(),1,-1.123676e+06,126949.145165
35,KNeighborsRegressor(),2,-1.125108e+06,129839.136125
39,KNeighborsRegressor(),3,-1.126061e+06,130404.851991
43,KNeighborsRegressor(),4,-1.126231e+06,135204.386175
47,KNeighborsRegressor(),5,-1.127491e+06,133871.425040
51,KNeighborsRegressor(),6,-1.129670e+06,135637.886959
27,KNeighborsRegressor(),7,-1.131488e+06,130244.621240
55,KNeighborsRegressor(),8,-1.133891e+06,134822.001339
59,KNeighborsRegressor(),9,-1.136415e+06,138595.531586
30,KNeighborsRegressor(),10,-1.137678e+06,126618.268687


I'll not keep the KNN because the model has a mean error of almost $1'000'000\$ ± 126'949 \$$  

### Tree based models

#### Random Forest

In [106]:
pipe_trees = Pipeline([
    ('model', RandomForestRegressor())
])

trees_grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 7],
    "model__min_samples_leaf": [1, 3],
    "model__max_features": ["sqrt", "log2"]
}

scoring = {
    "r2_score" : make_scorer(r2_score, greater_is_better=True),
    "neg_mae" : make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_mse" : make_scorer(mean_squared_error, greater_is_better=False),
    "neg_rmse" : make_scorer(root_mean_squared_error, greater_is_better=False),
    "neg_mape" : make_scorer(mean_absolute_percentage_error, greater_is_better=False),
}

In [130]:
grid_trees = GridSearchCV(
    estimator=pipe_trees,
    param_grid = trees_grid,
    scoring = scoring["neg_rmse"],
    cv = KFold(3),
    n_jobs = 11,
    verbose = 3
)

In [111]:
if os.path.isfile("../artifacts/grid_trees.pkl"):
    print("la grille existe déjà")
else : 
    grid_trees.fit(X_train, Y_train).fit(X_train, Y_train)
    dump(grid_trees, "../artifacts/grid_trees.pkl")

la grille existe déjà


In [131]:
grid_trees = load("../artifacts/grid_trees.pkl")

In [132]:
grid_trees.best_params_

{'model__max_depth': None,
 'model__max_features': 'sqrt',
 'model__min_samples_leaf': 1,
 'model__min_samples_split': 2,
 'model__n_estimators': 300}

In [133]:
best_model_trees = grid_trees.best_estimator_

y_pred = best_model_trees.predict(X_test)

print("R²:", r2_score(Y_test, y_pred))
print("MAE:", mean_absolute_error(Y_test, y_pred))
print("RMSE:", root_mean_squared_error(Y_test, y_pred))
print("MAPE:", mean_absolute_percentage_error(Y_test, y_pred))

R²: 0.8004904091975895
MAE: 298993.2185718967
RMSE: 773113.0387646062
MAPE: 0.1839464722982706


RandomForest() model gives us a better r2_score (0.80) and a mean absolute error of 298'993$, which is less than other linear models.

#### DecisionTreeRegressor

In [136]:
pipe_dtree = Pipeline([
    ('model', DecisionTreeRegressor())
])

dtree_grid = {
    "model__max_depth": [None, 10, 20, 50],
    "model__min_samples_split": [2, 7, 10],
    "model__min_samples_leaf": [1, 3, 10],
    "model__max_features": ["sqrt", "log2"]
}

scoring = {
    "r2_score" : make_scorer(r2_score, greater_is_better=True),
    "neg_mae" : make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_mse" : make_scorer(mean_squared_error, greater_is_better=False),
    "neg_rmse" : make_scorer(root_mean_squared_error, greater_is_better=False),
    "neg_mape" : make_scorer(mean_absolute_percentage_error, greater_is_better=False),
}

In [138]:
grid_dtree = GridSearchCV(
    estimator=pipe_dtree,
    param_grid = dtree_grid,
    n_jobs = 11,
    verbose = 3
)

In [143]:
if os.path.isfile("../artifacts/grid_dtree.pkl"):
    print("la grille existe déjà")
else : 
    grid_dtree.fit(X_train, Y_train).fit(X_train, Y_train)
    dump(grid_dtree, "../artifacts/grid_dtree.pkl")

la grille existe déjà


In [ ]:
grid_dtrees = load("./artifacts/grid_dtree.pkl")

In [140]:
grid_dtree.best_params_

{'model__max_depth': 50,
 'model__max_features': 'sqrt',
 'model__min_samples_leaf': 10,
 'model__min_samples_split': 7}

In [142]:
best_model_dtree = grid_dtree.best_estimator_

y_pred = best_model_dtree.predict(X_test)

print("R²:", r2_score(Y_test, y_pred))
print("MAE:", mean_absolute_error(Y_test, y_pred))
print("RMSE:", root_mean_squared_error(Y_test, y_pred))
print("MAPE:", mean_absolute_percentage_error(Y_test, y_pred))

R²: 0.6627096114815287
MAE: 423875.1682525371
RMSE: 1005224.5619045743
MAPE: 0.26442794232463057


### Conclusion 

I'll train tree-based models because they perform better than Linear Models when it comes to capture the links between the features.